In [4]:
from google.colab import drive, files
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import shutil
shutil.copy('MM-MIA.rar', '/content/drive/MyDrive/MM-MIA.rar')

# Extract directly to Drive
!apt-get install -q unrar
!unrar x /content/drive/MyDrive/MM-MIA.rar /content/drive/MyDrive/


FileNotFoundError: [Errno 2] No such file or directory: 'MM-MIA.rar'

In [7]:
# Vérifier la structure
!ls /content/MM-MIA/


ls: cannot access '/content/MM-MIA/': No such file or directory


In [8]:
# Installer les dépendances
!pip install -q iterative-stratification tokenizers transformers



In [9]:
# Corriger le chemin directement dans Colab (sans re-uploader)
script = "/content/drive/MyDrive/MM-MIA/Text classification/scripts/pretrain.py"

with open(script, 'r') as f:
    content = f.read()

content = content.replace(
    'PRETRAIN_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset/mimic_cxr_reports.csv"',
    'PRETRAIN_PATH = "/content/drive/MyDrive/dataset_labeled.csv"'
).replace(
    'TEXT_COL      = "reports"',
    'TEXT_COL      = "findings"'
)

with open(script, 'w') as f:
    f.write(content)

print("Fichier corrigé ✓")


Fichier corrigé ✓


In [10]:
  # Dans Colab, avant de lancer train.py
  !python "/content/drive/MyDrive/MM-MIA/Text classification/scripts/pretrain.py" \
      --csv /content/drive/MyDrive/dataset_labeled.csv \
      --text_col findings



MIMIC-CXR : 6473 rapports | Device : cuda
[00:00:00] Tokenize words                 ██████████████████ 2182     /     2182
[00:00:00] Count pairs                    ██████████████████ 2182     /     2182
[00:00:00] Compute merges                 ██████████████████ 4279     /     4279
Vocab     : 4359 tokens
Exemple   : ['[CLS]', 'Heart', 'Ġsize', 'Ġnormal', '.', 'ĠLungs', 'Ġare', 'Ġclear', '.', 'ĠXXXX', 'Ġare', 'Ġnormal']...
Params    : 5,462,023
  Epoch  1/20 | loss=4.6758 | acc=24.3% | lr=3.0e-04
  Epoch  2/20 | loss=2.6376 | acc=55.2% | lr=2.9e-04
  Epoch  3/20 | loss=1.9604 | acc=65.2% | lr=2.8e-04
  Epoch  4/20 | loss=1.6141 | acc=70.3% | lr=2.7e-04
  Epoch  5/20 | loss=1.4534 | acc=72.6% | lr=2.6e-04
  Epoch  6/20 | loss=1.2996 | acc=74.9% | lr=2.4e-04
  Epoch  7/20 | loss=1.1872 | acc=76.7% | lr=2.2e-04
  Epoch  8/20 | loss=1.1107 | acc=77.9% | lr=2.0e-04
  Epoch  9/20 | loss=1.0549 | acc=78.8% | lr=1.7e-04
  Epoch 10/20 | loss=0.9871 | acc=80.0% | lr=1.5e-04
  Epoch 11/20 | los

In [11]:
from google.colab import files

files.download("/content/drive/MyDrive/MM-MIA/Text classification/checkpoints/bert_pretrained.pt")
files.download("/content/drive/MyDrive/MM-MIA/Text classification/checkpoints/tokenizer.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
import os
print(os.path.exists("/content/drive/MyDrive/MM-MIA/multimodal_fusion/train.py"))


True


In [13]:
import shutil
shutil.copytree("/content/drive/MyDrive/MM-MIA", "/content/MM-MIA", dirs_exist_ok=True)


KeyboardInterrupt: 

In [14]:
# Run this in a separate cell while training is stopped
import shutil
print("Copying images to local disk...")
shutil.copytree("/content/drive/MyDrive/Png", "/content/Png")
print("Done.")



Copying images to local disk...
Done.


In [15]:
## Essaye de l'entraîner sur les gpus de télécom paris !!
!python /content//drive/MyDrive/MM-MIA/multimodal_fusion/train.py \
    --image_dir /content/Png \
    --csv /content/drive/MyDrive/dataset_labeled.csv \
    --vit_checkpoint /content/drive/MyDrive/best_vit_chest_04.pth \
    --batch_size 16


Device : cuda
Image dir : /content/Png
Dataset total : 6473 images
Train: 4529  Val: 976  Test: 968
Text encoder chargé : 4,275,712 params
config.json: 100% 790/790 [00:00<00:00, 4.55MB/s]
model.safetensors: 100% 343M/343M [00:24<00:00, 14.2MB/s]
Loading weights: 100% 198/198 [00:00<00:00, 4892.12it/s]
[transformers] ViTModel LOAD REPORT from: codewithdark/vit-chest-xray
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
ViT checkpoint chargé — missing: 192, unexpected: 192
Modèle total : 95,931,925 paramètres
Phase 1 — entraînables : 5,266,965 (cross-attn + tête)
Epoch 01/30 | LR

In [17]:
import sys, os
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from tokenizers import Tokenizer
from sklearn.metrics import roc_auc_score

sys.path.insert(0, "/content/drive/MyDrive/MM-MIA/multimodal_fusion")
sys.path.insert(0, "/content/drive/MyDrive/MM-MIA/Text classification")

from models  import BERTForMLM
from model   import MultimodalFusion
from dataset import MultimodalDataset, multimodal_collate, LABEL_COLS, load_paired_df
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

DEVICE    = torch.device("cuda")
TEXT_CKPT = "/content/drive/MyDrive/MM-MIA/Text classification/checkpoints"
CKPT_PATH = "/content/drive/MyDrive/MM-MIA/multimodal_fusion/checkpoints/multimodal_fusion.pt"
CSV_PATH  = "/content/drive/MyDrive/dataset_labeled.csv"
IMG_DIR   = "/content/Png"


In [18]:
# ── Rebuild test split (same seed = same split) ──────────────────────────────
df = load_paired_df(CSV_PATH)
msss  = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
_, temp_idx = next(msss.split(df, df[LABEL_COLS]))
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(msss2.split(df.iloc[temp_idx], df.iloc[temp_idx][LABEL_COLS]))
test_df = df.iloc[temp_idx].iloc[test_idx].reset_index(drop=True)

VAL_TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5]),
])
tok      = Tokenizer.from_file(os.path.join(TEXT_CKPT, "tokenizer.json"))
test_ds  = MultimodalDataset(test_df, LABEL_COLS, tok, IMG_DIR, VAL_TF)
test_loader = DataLoader(test_ds, 16, shuffle=False,
                         collate_fn=multimodal_collate, num_workers=2)

# ── Load model ───────────────────────────────────────────────────────────────
from transformers import ViTModel
bert = BERTForMLM(tok.get_vocab_size(), 256, 8, 6, 512)
bert.load_state_dict(torch.load(os.path.join(TEXT_CKPT, "bert_pretrained.pt"),
                                map_location=DEVICE, weights_only=True))
vit   = ViTModel.from_pretrained("codewithdark/vit-chest-xray")
model = MultimodalFusion(text_encoder=bert.encoder, vit=vit,
                         n_labels=len(LABEL_COLS), d_model=512,
                         n_heads=8, dropout=0.1).to(DEVICE)
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True))
model.eval()

# ── Collect predictions ──────────────────────────────────────────────────────
all_probs, all_labels = [], []
with torch.no_grad():
    for ids, pixels, labels in test_loader:
        logits = model(ids.to(DEVICE), pixels.to(DEVICE))
        all_probs.append(logits.sigmoid().cpu().numpy())
        all_labels.append(labels.numpy())

probs = np.vstack(all_probs)
gt    = np.vstack(all_labels)

# ── Per-class AUC ────────────────────────────────────────────────────────────
print(f"\n{'Pathology':<35} {'AUC':>6}")
print("-" * 43)
aucs = []
for i, label in enumerate(LABEL_COLS):
    if len(np.unique(gt[:, i])) > 1:
        auc = roc_auc_score(gt[:, i], probs[:, i])
        aucs.append(auc)
        print(f"{label:<35} {auc:.4f}")
    else:
        print(f"{label:<35}  N/A  (single class in test)")
print("-" * 43)
print(f"{'Mean AUC':<35} {np.mean(aucs):.4f}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: codewithdark/vit-chest-xray
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Pathology                              AUC
-------------------------------------------
Atelectasis                         0.9875
Cardiomegaly                        0.9887
Effusion                            0.9731
Pneumonia                           0.9670
Pneumothorax                        0.9969
Edema                               0.9922
Emphysema                           0.9604
Fibrosis                            0.9550
Infiltration                        0.9839
Mass                                0.9946
Nodule                              0.9918
Hernia                              0.9997
Fracture                            0.9922
Pleural_Thickening                  1.0000
Opacity                             0.9999
Consolidation                       0.9936
Granuloma                           0.9999
Calcinosis                          0.9974
Scoliosis                           0.9950
Atherosclerosis                     1.0000
Normal                              0.9920
---------